In [0]:
#Databricks EDA notebook source
# MAGIC %md
# MAGIC ## RetailRocket Dataset - Exploratory Data Analysis
# MAGIC
# MAGIC **Goal:** Understand the raw data before building the ETL pipeline.
# MAGIC **Scope:** Full dataset from Azure Blob
# MAGIC **Key investigative questions:** What's in the data, what's the shape, what features exist, what patterns exist?

# COMMAND ----------
storage_key = dbutils.secrets.get(scope = "retailrocket", key="storage-key")
spark.conf.set("fs.azure.account.key.stdportfolio.blob.core.windows.net", storage_key) #combined string 'storage account' + this resource + use key value

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import LongType

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Data Overview

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1.1 Load RetailRocket CSVs

# COMMAND ----------

events = (
    spark.read
    .option("header","true")
    .option("inferSchema", "false")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/events.csv")
)

category_tree = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/category_tree.csv")
)

props_part1 = (
    spark.read
    .option("header","true")
    .option("inferSchema", "true")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/item_properties_part1.csv")
)

props_part2 = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/item_properties_part2.csv")
)

item_properties = props_part1.union(props_part2)

print("All files loaded.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1.2 Row counts

# COMMAND ----------

print(f"Events: {events.count():>10,} rows")
print(f"Category tree: {category_tree.count():>10,} rows")
print(f"Item Properties: {item_properties.count():>10,} rows")


# COMMAND ----------

print("=== Events ===")
events.printSchema()

print("\n=== Category Tree ===")
category_tree.printSchema()

print("\n=== Item Properties ===")
item_properties.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1.4 Unique Counts

# COMMAND ----------

print(f"Unique visitors: {events.select('visitorid').distinct().count():>10,}")
print(f"Unique items: {item_properties.select('itemid').distinct().count():>10,}")
print(f"Unique categories: {category_tree.select('categoryid').distinct().count():>10,}")

# COMMAND ----------



In [0]:
# MAGIC %md
# MAGIC ## 2. Data Quality

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.1 Null counts per column

# COMMAND ----------

print("=== Events - Column Nulls ===")
for col in events.columns:
    null_count = events.filter(F.col(col).isNull()).count()
    print(f" {col:20} {null_count:>10,} nulls ({null_count/events.count()*100:.1f}%)")

# COMMAND ----------

print("\n === Category Tree - Column Nulls ===")
for col in category_tree.columns:
    null_count = category_tree.filter(F.col(col).isNull()).count()
    print(f" {col:20} {null_count:>10,} nulls ({null_count/category_tree.count()*100:.1f}%)")

# COMMAND ----------

print("\n === Item Properties - Column Nulls ===")
for col in item_properties.columns:
    null_count = item_properties.filter(F.col(col).isNull()).count()
    print(f" {col:20} {null_count:>10,} nulls ({null_count/item_properties.count()*100:.1f}%)")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.2 Duplicate check

# COMMAND ----------

events_total = events.count()
events_distinct = events.distinct().count()
print(f"Events total: {events_total:,}")
print(f"Distinct events: {events_distinct:,}")
print(f"Duplicates: {events_total - events_distinct}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2.3 Data types and sample (understanding)

# COMMAND ----------

events.printSchema()
display(events.limit(5))

# COMMAND ----------

category_tree.printSchema()
display(category_tree.limit(5))

# COMMAND ----------

item_properties.printSchema()
display(item_properties.limit(5))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Event Distribution

# COMMAND ----------

# MAGIC %md
# MAGIC ### 3.1 Event type breakdown

# COMMAND ----------

event_dist = (
    events
    .groupBy("event")
    .agg(F.count("*").alias("count"))
    .withColumn("pct", F.col("count") / events.count()*100) # Same precedence
    .orderBy(F.desc("count"))
)

display(event_dist)

# COMMAND ----------

views = events.filter(F.col("event") == "view").count()
carts = events.filter(F.col("event") == "addtocart").count()
purchases = events.filter(F.col("event") == "transaction").count()

print(f"Conversion Funnel:")
print(f" Views: {views:>10,} (100%)")
print(f" Add to Cart: {carts:>10,} ({carts/views*100:.2f})% of views")
print(f" Transactions: : {purchases:>10,} ({purchases/views*100:.2f})% of views")
print(f" Cart -> Purchase: {purchases/carts*100:.2f}% of carts.") #Conversion %



In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## 4. User Analysis

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4.1 Unique Users

# COMMAND ----------

unique_users = events.select("visitorid").distinct().count()
print(f"Unique users: {unique_users:,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4.2 Events/User

# COMMAND ----------

user_events = (
    events
    .groupBy("visitorid")
    .agg(F.count("*").alias("event_count"))
)

display(user_events.drop("visitorid").describe())

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4.3 User Activity Segments

# COMMAND ----------
# Check percentiles
percentiles = user_events.approxQuantile("event_count", [0.25, 0.50, 0.75, 0.90, 0.95, 0.99], 0.01)
print(f"25th: {percentiles[0]}")
print(f"50th: {percentiles[1]}")
print(f"75th: {percentiles[2]}")
print(f"90th: {percentiles[3]}")
print(f"95th: {percentiles[4]}")
print(f"99th: {percentiles[5]}")

single_event =user_events.filter(F.col("event_count")==1).count()
low_activity = user_events.filter((F.col("event_count") >=2) & (F.col("event_count") <=3)).count()
medium_activity  = user_events.filter((F.col("event_count") >=4) & (F.col("event_count") <=5)).count()
high_activity  = user_events.filter(F.col("event_count") >=6).count()

print(f"\n=== User Activity Segment ===")
print(f"Single event (1): {single_event:,} ({single_event/unique_users*100:.2f})% of Users")
print(f"Low Activity (2-3): {low_activity:,} ({low_activity/unique_users*100:.2f})% of Users.")
print(f"Medium Activity (4-5): {medium_activity:,} ({medium_activity/unique_users*100:.2f})% of Users")
print(f"High Activity (6+): {high_activity:,} ({high_activity/unique_users*100:.2f})% of Users")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Bot Analysis, Events Per Day
# Calculate events per day for each user
user_velocity = (
    events
    .withColumn("timestamp_ms", F.col("timestamp").cast(LongType()))
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp_ms") / 1000))
    .groupBy("visitorid")
    .agg(
        F.count("*").alias("total_events"),
        F.min("event_timestamp").alias("first_seen"),
        F.max("event_timestamp").alias("last_seen"),
        F.countDistinct(F.to_date("event_timestamp")).alias("active_days")
    )
    .withColumn("events_per_day", F.col("total_events") / F.col("active_days"))
)

# Users with >50 events per day are likely bots
bots_velocity = user_velocity.filter(F.col("events_per_day") > 50)
print(f"Number of Suspected Bots: {bots_velocity.count():,} ({bots_velocity.count()/unique_users*100:.2f})%")

# MAGIC %md
# MAGIC ### 4.4 Users who purchased vs didn't

# COMMAND ----------

users_who_purchased = (
    events
    .filter(F.col("event") == "transaction")
    .select("visitorid")
    .count()
)

users_only_browsed = unique_users - users_who_purchased

#Users, not events.
print(f"Users who only browsed but didnt purchase: {users_only_browsed:,} ({users_only_browsed/unique_users*100:.2f})% of users")


## 5. Item Analysis

In [0]:

# MAGIC ### 5.1 Unique Items

unique_items = events.select("itemid").distinct().count()
print(f"Unique items: {unique_items:,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.2 Most Viewed Items

# COMMAND ----------

top_viewed = (
    events
    .filter(F.col("event") == "view")
    .groupBy("itemid")
    .agg(F.count("*").alias("views"))
    .orderBy(F.desc("views"))
)

display(top_viewed.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.3 Most Purchased Items

# COMMAND ----------

top_purchased = (
    events
    .filter(F.col("event") == "transaction")
    .groupBy("itemid")
    .agg(F.count("*").alias("purchases"))
    .orderBy(F.desc("purchases"))
)

display(top_purchased.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.4 Items viewed but never purchased

# COMMAND ----------

viewed_items = events.filter(F.col("event") == "view").select("itemid").distinct()
purchased_items = events.filter(F.col("event") == "transaction").select("itemid").distinct()

never_purchased = viewed_items.subtract(purchased_items).count()

## 6. Temporal Patterns

In [0]:
# MAGIC ### 6.1 Add time components
events_with_time = (
    events
    .withColumn("timestamp_ms", F.col("timestamp").cast(LongType()))
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp_ms") / 1000))
    .withColumn("hour", F.hour("event_timestamp"))
    .withColumn("minute", F.minute("event_timestamp"))
    .withColumn("day_of_week", F.dayofweek("event_timestamp"))
    .withColumn("date", F.to_date("event_timestamp"))
    .withColumn("month", F.month("event_timestamp"))
)

display(events_with_time.select("event_timestamp","hour","day_of_week","date","month").limit(10))

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6.2 Daily event volume
# COMMAND ----------

print(f"Daily Events Snapshot")
daily = (
    events_with_time
    .groupBy("date")
    .agg(F.count("*").alias("events"))
    .orderBy("date")
)

display(daily.limit(10), "Daily Events Snapshot")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6.3 Hourly Distribution
# COMMAND ----------

print(f"Hourly Distribution")
hourly = (
    events_with_time
    .groupBy("hour")
    .agg(F.count("*").alias("events"))
    .orderBy("hour")
)

display(hourly.limit(10), "Hourly Distribution")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6.4 Weekday and Weekend Analysis
# COMMAND ----------

print(f"Weekday Distribution")
weekday_dist = (
    events_with_time
    .withColumn("day_type", F.when(F.col("day_of_week").isin(1,7), "Weekend").otherwise("Weekday")) #If saturday/sunday = weekend, else weekday.
    .groupBy("day_type")
    .agg(F.count("*").alias("events"))
    .orderBy(F.desc("events"))
)

display(weekday_dist, "Weekday Distribution")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6.5 Conversion by hour
# COMMAND ----------

print(f"Conversion by Hour")
conversion_by_hour = (
    events_with_time
    .groupBy("hour")
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event") == "addtocart",1).otherwise(0)).alias("carts"),
        F.sum(F.when(F.col("event") == "transaction",1).otherwise(0)).alias("purchases"),
    )
    .withColumn("cart_conversion_rate", F.round(F.col("carts")/F.col("views")*100, 2))
    .withColumn("purchase_fromcart_conversion_rate", F.round(F.col("purchases")/F.col("carts")*100, 2))
    .withColumn("purchase_conversion_rate", F.round(F.col("purchases")/F.col("views")*100, 2))
    .orderBy(F.desc("purchase_conversion_rate"))
)

display(conversion_by_hour.limit(10), "Conversion by hour")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6.6 Conversion by day
# COMMAND ----------

conversion_by_day = (
    events_with_time
    .groupBy("day_of_week")
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event")=="view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event")=="addtocart",1).otherwise(0)).alias("carts"),
        F.sum(F.when(F.col("event")=="transaction",1).otherwise(0)).alias("purchases")
    )
    .withColumn("cart_conversion_rate", F.round(F.col("carts")/F.col("views")*100,2))
    .withColumn("purchase_from_cart_conversion_rate", F.round(F.col("purchases")/F.col("carts")*100,2))
    .withColumn("purchase_conversion_rate", F.round(F.col("purchases")/F.col("views")*100,2))
    .orderBy("day_of_week")
)

print(f"Conversion by Day of the Week")
display(conversion_by_day.limit(10))



## 7. Category Analysis

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ### 7.1 Category tree structure
# COMMAND ----------
from pyspark.sql.window import Window

#Total Categories
print(f"Total categories: {category_tree.count():,}")
#Root Categories
print(f"Root categories: {category_tree.filter(F.col("parentid").isNull()).count():,}")
#Child categories
print(f"Child categories: {category_tree.filter(F.col("parentid").isNotNull()).count():,}")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 7.2 Item properties — categoryid extraction
# COMMAND ----------

#Filter for categoryid, values are hashed for data privacy by data provider
category_props = item_properties.filter(F.col("property") == "categoryid")
print(f"Category property rows: {category_props.count():,}")

#Unique items with category
items_with_category = category_props.select("itemid").distinct().count()
print(f"Items with category: {items_with_category:,}")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 7.3 Top categories by event volume
# COMMAND ----------

#Latest category per item (historical change)
print(category_props.columns)
category_props.printSchema()

latest_category = (
    category_props
    .withColumn("timestamp_ms", F.col("timestamp").cast(LongType())) #Change data type.
    .withColumn("row_number", F.row_number().over(  # Create new column row number
        Window.partitionBy("itemid").orderBy(F.desc("timestamp_ms")) # Group by item, sort by latest timestamp.
    ))
    .filter(F.col("row_number") == 1) #Select newest assigned category only
    .select("itemid", F.col("value").cast(LongType()).alias("categoryid"))
)

display(latest_category.limit(5)) #Check


#Join with events
events_with_category=(
    events
    .join(latest_category,"itemid","left")
)

top_categories = (
    events_with_category
    .filter(F.col("categoryid").isNotNull())
    .groupBy("categoryid")
    .agg(F.count("*").alias("total_events"))
    .orderBy(F.desc("total_events"))
)

display(top_categories.limit(15))



## Exploratorary Summary

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## 8. Summary
# COMMAND ----------
print("=" * 60)
print("RETAILROCKET DATASET SUMMARY")
print("=" * 60)
print(f"Date range:        May 3 – September 18, 2015 (138 days)")
print(f"Total events:      {events.count():,}")
print(f"Unique users:      {unique_users:,}")
print(f"Unique items:      {unique_items:,}")
print(f"Categories:        {category_tree.count():,}")
print("-" * 60)
print(f"Views:             {views:,}")
print(f"Carts:             {carts:,}")
print(f"Purchases:         {purchases:,}")
print("-" * 60)
print(f"View→Cart rate:    {carts/views*100:.2f}%")
print(f"Cart→Purchase:     {purchases/carts*100:.2f}%")
print(f"Overall conversion: {purchases/views*100:.2f}%")
print(f"Most active conversion day: {conversion_by_day.orderBy(F.desc("purchase_conversion_rate")).limit(1).first()['day_of_week']}th day of the week with {conversion_by_day.orderBy(F.desc("purchase_conversion_rate")).limit(1).first()['purchase_conversion_rate']}% conversion rate.")
print("-" * 60)
print(f"Users who bought:  {users_who_purchased:,} ({users_who_purchased/unique_users*100:.1f}%)")
print(f"Users who browsed: {users_only_browsed:,} ({users_only_browsed/unique_users*100:.1f}%)")
print("=" * 60)